In [1]:
import chromadb
from collections import Counter

client = chromadb.PersistentClient(
    path="/Users/tonylung/Desktop/FEED-SYSTEM-GAI/databases/hiwin_vector_db"
)

col = client.get_collection("screw_specs")
data = col.get(include=["metadatas"])

triples = Counter(
    (
        m.get("brand", ""),
        m.get("category", ""),
        m.get("data_type", "")
    )
    for m in data["metadatas"]
)

print("--- 資料類型統計 ---")
for key, count in sorted(triples.items(), key=lambda x: x[1], reverse=True):
    print(f"{key}: {count}")


--- 資料類型統計 ---
('FANUC', 'Motor', 'MotorDetail'): 2196
('PMI', 'Screw', 'Specification'): 1638
('HIWIN', 'Screw', 'Specification'): 1168
('FANUC', 'Manual', 'Manual'): 290
('PMI', 'Manual', 'Manual'): 124
('FANUC', 'Motor', 'MotorModel'): 93
('HIWIN', 'Manual', 'Manual'): 87


In [4]:
import chromadb
from chromadb.utils import embedding_functions
import torch
# 1. 重新呼叫翻譯官 
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3",
    device="cuda"  # 確保調用你的 RTX 4060
)
print(f"CUDA 是否可用: {torch.cuda.is_available()}")
# 2. 連接到硬碟裡的資料庫
client = chromadb.PersistentClient(path="./hiwin_vector_db")

# 3. 取得當初存入 Chunks 的 Collection
manual_collection = client.get_collection(name = "screw_manuals", embedding_function = emb_fn)
spec_collection = client.get_collection(name = "hiwin_specs", embedding_function = emb_fn)


print("成功連接向量資料庫，Chunks 已準備好被檢索！")

CUDA 是否可用: True
成功連接向量資料庫，Chunks 已準備好被檢索！


In [ ]:
import ollama

# 定義專業知識字典
SERIES_INFO = {
    "FDC": "雙螺帽設計，具備極高的軸向剛性與預壓穩定性，專為重負荷精密工具機設計。",
    "FSW": "小法蘭單螺帽設計，體積精簡，適合安裝空間受限的自動化設備。",
    "FSV": "標準單螺帽型，具備優異的傳動效率與流暢度，是自動化產業最泛用的標準件。",
    "RSI": "旋轉螺帽設計，適合絲槓固定、螺帽旋轉的機構，能有效抑制長行程下的振動。",
    "FSI": "內循環設計，螺帽外徑小，運轉安靜，適合小型精密設備。"
}

def get_expert_advice(user_query, calc_result, use_rag=True):
    """
    混合檢索架構：同時檢索技術手冊 (Manual) 與 產品規格 (Specs)
    """
    rag_context = ""
    rag_status_msg = ""

    # --- RAG 混合檢索邏輯 ---
    if use_rag:
        try:
            # A. 檢索【技術手冊】(Manual Chunks): 找潤滑、安裝、原理、壽命
            manual_res = manual_collection.query(query_texts=[user_query], n_results=2)
            manual_text = "\n【技術手冊參考資料】：\n" + "\n".join(manual_res['documents'][0])
            
            # B. 檢索【產品規格】(Specs Documents): 找替代型號、詳細尺寸、參數對比
            spec_res = spec_collection.query(query_texts=[user_query], n_results=3)
            spec_text = "\n【相似型號規格參考】：\n" + "\n".join(spec_res['documents'][0])
            
            # 整合兩路檢索結果
            rag_context = f"{spec_text}\n{manual_text}"
            rag_status_msg = "\n(系統提示：已完成混合檢索 - 參考手冊與規格表)\n"
            
        except Exception as e:
            rag_context = f"\n(系統提示：資料庫檢索失敗: {e})\n"
    
    # --- 數據準備 (目前計算出的最優解) ---
    series = calc_result.get('series', '標準')
    model = calc_result.get('model', '未知')
    feature = SERIES_INFO.get(series, "HIWIN 精密傳動元件。")
    
    spec_context = f"""
    【目前推薦型號數據】
    - 推薦系列：{series} ({feature})
    - 具體型號：{model}
    - 物理參數：公稱外徑 {calc_result['dia']}mm, 導程 {calc_result['lead']}mm
    - 動負荷能力：{calc_result['dynamic_load']} kgf
    """
    
    # --- 組合最終 Prompt ---
    # 這裡我們明確區分「目前數據」與「參考資料」，幫助 LLM 進行對比分析
    prompt = f"""
    你是一位專業的 HIWIN 技術支援工程師，請根據提供的【目前推薦型號數據】與【參考資料】來回答提問。
    回答時請結合產品的物理特性（如外徑、負荷）與系列優點（如剛性、空間利用)，請用繁體中文回答。
       
    當使用者詢問關於空間、尺寸或替代型號時，請優先參考「相似型號規格」進行對比。
    當使用者詢問關於安裝、保養或技術原理時，請參考「技術手冊參考資料」。

    {spec_context}
    
    {rag_context}
    
    使用者提問：{user_query}
    
    請用繁體中文回答，語氣專業且誠懇，並儘可能引用具體參數。
    """
    
    # --- 呼叫 Qwen 2.5 ---
    try:
        response = ollama.generate(
            model='qwen2.5:7b', 
            prompt=prompt,
            options={"temperature": 0.3} 
        )
        return response['response'] + rag_status_msg
    except Exception as e:
        return f"連線 Ollama 發生錯誤: {str(e)}"
    


In [7]:
import ollama
import chromadb

# 設定資料庫路徑 (自動識別系統環境)
import platform
DB_PATH = "/Users/tonylung/Desktop/FEED-SYSTEM-GAI/databases/hiwin_vector_db"
if platform.system() == "Windows":
    DB_PATH = DB_PATH.replace("/Users/tonylung", "C:/Users/YourUserName") # 請根據實際 Windows 路徑修改

client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_collection("screw_specs")

def get_expert_advice(user_query, calc_result):
    """
    針對 Sigma CNC 機械研發場景優化的 RAG 邏輯
    """
    
    # --- 1. 精準檢索策略 ---
    # 同時檢索規格 (Specs) 與 手冊 (Manual)，並加入品牌過濾 (可選)
    def smart_query(query, data_type, n=2):
        return collection.query(
            query_texts=[query],
            n_results=n,
            where={"data_type": data_type} # 核心優化：利用你的統計類別進行過濾
        )

    try:
        spec_res = smart_query(user_query, "Specification", n=3)
        manual_res = smart_query(user_query, "Manual", n=2)
        
        rag_context = f"""
【技術規格參考】:
{" ".join(spec_res['documents'][0])}

【手冊操作細節】:
{" ".join(manual_res['documents'][0])}
        """
    except Exception as e:
        rag_context = f"檢索異常: {e}"

    # --- 2. 建立 Prompt (針對 Gemma 3 4B 優化) ---
    # Gemma 3 對 XML 標籤或結構化區塊反應極佳
    prompt = f"""
    <Role>你是一位精通機械傳動與 CNC 系統的資深工程師，服務於 Sigma CNC Technology。</Role>
    
    <Context>
    使用者目前計算出的建議型號：
    - 品牌系列：{calc_result.get('brand')} {calc_result.get('series')}
    - 型號：{calc_result.get('model')}
    - 物理參數：直徑 {calc_result.get('dia')}mm / 導程 {calc_result.get('lead')}mm
    - 動負荷：{calc_result.get('dynamic_load')} kgf
    </Context>

    <Reference_Data>
    {rag_context}
    </Reference_Data>

    <User_Query>
    {user_query}
    </User_Query>

    <Instruction>
    1. 請分析建議型號是否滿足使用者需求。
    2. 參考 Reference_Data 中的技術細節（如安裝注意、潤滑要求或剛性表現）。
    3. 若涉及到 FANUC 電機匹配，請結合資料庫中的 Motor 資訊。
    4. 請使用「繁體中文」回答，保持專業工程師口吻，必要時以條列式說明。
    </Instruction>
    """

    # --- 3. 執行生成 ---
    try:
        response = ollama.generate(
            model='gemma3n:e4b',
            prompt=prompt,
            options={
                "temperature": 0.3, # 保持技術準確性
                "num_ctx": 4096     # 確保 Context 夠大處理 RAG 內容
            }
        )
        return response['response']
    except Exception as e:
        return f"Ollama Error: {e}"

# --- 測試案例 ---
# if __name__ == "__main__":
#     模擬從計算模組傳入的結果
#     current_calc = {
#         "brand": "HIWIN",
#         "series": "FDC",
#         "model": "R40-10B2-FDC",
#         "dia": 40,
#         "lead": 10,
#         "dynamic_load": 4850
#     }
    
#     query = "這個型號的 FDC 螺帽在安裝時有什麼特別需要注意的地方？跟 PMI 的同等級產品比起來呢？"
#     print(get_expert_advice(query, current_calc))

In [ ]:
#模型:Ollama qwen2.5:7b，無RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "brand": "HIWIN",
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議"

print("正在調用 Qwen 2.5:7b 進行分析...\n")
result = get_expert_advice(user_query, calc_result, use_rag = False)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 Qwen 2.5:7b 進行分析...



您好，

感謝您對HIWIN產品的興趣。關於您的問題，根據目前推薦的FDC系列螺帽設計及其物理特性，我們可以提供以下建議：

1. **型號40-12K5**：此型號具有公稱外徑 40.0mm 和導程 12.0mm 的特點。其軸向剛性與預壓穩定性極高，適合重負荷精密工具機使用。動負荷能力為7430 kgf。

若考慮螺帽直徑和長度空間有限的問題，我們可以參考相似型號規格進行對比：

- **型號40-10K5**：此型號與40-12K5相比，導程較短（10.0mm），外徑相同。雖然軸向剛性與預壓穩定性可能會稍有下降，但仍然適合中等負荷應用。

- **型號36-12K5**：此型號的公稱外徑減小為36.0mm，導程仍保持在12.0mm。這可能更符合您提到的空間限制需求，同時仍能提供良好的軸向剛性和預壓穩定性。

以上建議皆基於相似型號規格進行比較，具體選擇需根據您的負荷需求和實際安裝空間來決定。我們建議在確定最終選型前，進一步確認您的具體應用條件。

如有更多技術問題或需要更詳細的資料，歡迎隨時聯繫我們。感謝您的信任與支持！

敬上

In [ ]:
#模型:Ollama qwen2.5:7b，有RAG回答

# 模擬你公式跑完後的結果 (Dictionary 格式)
calc_result = {
    "series": "FDC",
    "model": "40-12K5",
    "dia": 40.0,
    "lead": 12.0,
    "dynamic_load": 7430
}

# 測試提問
user_query = "螺帽直徑跟長度空間有限，是否有其他型號建議?"

print("正在調用 gemma3n:e4b 進行分析...\n")
result = get_expert_advice(user_query, calc_result)

from IPython.display import Markdown
Markdown(result) # 使用 Markdown 讓回答看起來更漂亮

正在調用 gemma3n:e4b 進行分析...



In [1]:
import chromadb
import time

# 設定資料庫路徑
DB_PATH = "/Users/tonylung/Desktop/FEED-SYSTEM-GAI/databases/hiwin_vector_db"

def run_retrieval_test():
    print("--- [RAG 檢索能力檢驗啟動] ---")
    
    # 1. 連接資料庫 (計算加載時間)
    start_time = time.time()
    client = chromadb.PersistentClient(path=DB_PATH)
    
    # 檢查 Collection 是否存在
    try:
        col = client.get_collection("screw_specs")
    except Exception as e:
        print(f"錯誤：找不到 collection 'screw_specs'。{e}")
        return
    
    print(f"資料庫載入耗時: {time.time() - start_time:.2f}s")

    # 2. 定義測試案例 (Test Cases)
    test_cases = [
        {
            "label": "技術規格檢索 (Specification)",
            "query": "FDC 40-10 螺帽的動負荷是多少？",
            "filter": {"data_type": "Specification"}
        },
        {
            "label": "安裝與維護檢索 (Manual)",
            "query": "滾珠螺帽如何進行潤滑保養？",
            "filter": {"data_type": "Manual"}
        },
        {
            "label": "電機匹配檢索 (MotorDetail)",
            "query": "FANUC αi 系列電機的轉矩參數",
            "filter": {"data_type": "MotorDetail"}
        }
    ]

    # 3. 執行測試
    for case in test_cases:
        print(f"\n[測試項目]: {case['label']}")
        print(f"  查詢關鍵字: {case['query']}")
        
        # 執行檢索
        search_start = time.time()
        results = col.query(
            query_texts=[case['query']],
            n_results=3,
            where=case['filter']
        )
        search_duration = (time.time() - search_start) * 1000 # 毫秒
        
        # 顯示結果摘要
        if results['documents'] and len(results['documents'][0]) > 0:
            print(f"  檢索耗時: {search_duration:.2f}ms")
            for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
                # 擷取前 100 字顯示
                snippet = doc[:100].replace('\n', ' ')
                brand = meta.get('brand', 'N/A')
                print(f"    Top {i+1} [{brand}]: {snippet}...")
        else:
            print("  [警告]: 未能檢索到相關資料，請檢查 Metadata 標籤是否正確。")

run_retrieval_test()

--- [RAG 檢索能力檢驗啟動] ---
資料庫載入耗時: 0.09s

[測試項目]: 技術規格檢索 (Specification)
  查詢關鍵字: FDC 40-10 螺帽的動負荷是多少？
  檢索耗時: 425.62ms
    Top 1 [PMI]: 這是銀泰 (PMI) 的滾珠螺桿規格。系列名稱為 FSDC，型號為 FSDC-40-10-BD6.35-C5。外徑為 40 mm，導程為 10 mm，鋼珠直徑為 6.35 mm，循環圈數為 5，動負荷...
    Top 2 [PMI]: 這是銀泰 (PMI) 的滾珠螺桿規格。系列名稱為 FSDC，型號為 FSDC-32-10-BD4.762-C5。外徑為 32 mm，導程為 10 mm，鋼珠直徑為 4.762 mm，循環圈數為 5，動...
    Top 3 [PMI]: 這是銀泰 (PMI) 的滾珠螺桿規格。系列名稱為 FDDC，型號為 FDDC-40-10-BD6.35-C5。外徑為 40 mm，導程為 10 mm，鋼珠直徑為 6.35 mm，循環圈數為 5，動負荷...

[測試項目]: 安裝與維護檢索 (Manual)
  查詢關鍵字: 滾珠螺帽如何進行潤滑保養？
  檢索耗時: 106.05ms
    Top 1 [PMI]: 綜合技術型錄 滾珠螺桿 線性滑軌 線性模組...
    Top 2 [HIWIN]: 矩將會產生熱及降低預期壽命。透過我們特別 的設計及製程，提供給您最佳化的滾珠螺桿  零背隙和低熱損失。 一般建議預壓力不超過8％動負荷C (10 6 revs)，若要更詳細資料請與HIWIN連繫。...
    Top 3 [PMI]: 我們承諾並致力推動下列環安衛政策：  一、遵循環安衛法規，致力污染預防，杜絕災害發生。  二、創新綠色研發，降低能源耗用，符合客戶需求。 三、強化風險管理，確保人身安全，提升環安衛績效。 四、全員參與...

[測試項目]: 電機匹配檢索 (MotorDetail)
  查詢關鍵字: FANUC αi 系列電機的轉矩參數
  檢索耗時: 122.23ms
    Top 1 [FANUC]: FANUC 伺服馬達型號 αiS 50/3000 FAN-D 的規格明細：Weight，符號 w，數值 46.0 kg。...
    Top 2 

In [9]:
import chromadb

def get_brand_models(brand_name):
    client = chromadb.PersistentClient(path="/Users/tonylung/Desktop/FEED-SYSTEM-GAI/databases/hiwin_vector_db")
    col = client.get_collection("screw_specs")
    
    # 僅檢索該品牌的 Metadata，不進行向量比對 (n_results 設大一點以獲取清單)
    results = col.get(
        where={"brand": brand_name},
        include=["metadatas"]
    )
    
    # 提取不重複的型號
    models = sorted(list(set(m.get("model_id") for m in results["metadatas"] if m.get("model_id"))))
    return models

# 測試：選擇 HIWIN 後，系統應回傳所有 HIWIN 型號
if __name__ == "__main__":
    target_brand = "HIWIN"
    available_models = get_brand_models(target_brand)
    print(f"--- {target_brand} 可選型號清單 (前 10 筆) ---")
    print(available_models[:10])

--- HIWIN 可選型號清單 (前 10 筆) ---
['100-10T6', '100-12B2', '100-12B3', '100-12T4', '100-12T6', '100-16B2', '100-16B3', '100-16T4', '100-16T6', '100-20B2']


In [8]:
import chromadb

def get_unique_models(brand_name: str):
    # 自動適應路徑 (Mac)
    db_path = "/Users/tonylung/Desktop/FEED-SYSTEM-GAI/databases/hiwin_vector_db"
    client = chromadb.PersistentClient(path=db_path)
    col = client.get_collection("screw_specs")
    
    print(f"--- 正在檢索品牌: {brand_name} ---")
    
    # 修正點：使用與 Embedding Code 一致的 brand 標籤
    # 限制條件：Specification 類別且品牌匹配
    results = col.get(
        where={
            "$and": [
                {"brand": brand_name},
                {"data_type": "Specification"}
            ]
        },
        include=["metadatas"]
    )
    
    if not results["metadatas"]:
        print("警告：未找到匹配資料，請檢查品牌名稱大小寫是否一致。")
        return []
        
    # 修正點：根據你的 Embedding Code，Key 是 'model_id'
    models = {m.get("model_id") for m in results["metadatas"] if m.get("model_id")}
    
    return sorted(list(models))

if __name__ == "__main__":
    # 測試單元
    hiwin_list = get_unique_models("HIWIN")
    print(f"成功抓取 {len(hiwin_list)} 筆型號。")
    if hiwin_list:
        print("前 5 筆型號：", hiwin_list[:5])

--- 正在檢索品牌: HIWIN ---
成功抓取 409 筆型號。
前 5 筆型號： ['100-10T6', '100-12B2', '100-12B3', '100-12T4', '100-12T6']
